In [ ]:
import pandas as pd
from pathlib import Path
import zipfile
import re
from IPython.display import display

date_str = "20250702"

# ==== 🔧 PATHS ====
mapping_excel = Path(f"D:/D&T Project/Planner Report/{date_str}/mapping_file_{date_str}.xlsx")
excel_dir = Path(f"D:/D&T Project/Planner Report/{date_str}/")
cluster_dir = Path("D:/D&T Project/Planner Report/PAT Report Approved")
output_dir = excel_dir

# ==== 🔁 Load Mapping ====
df = pd.read_excel(mapping_excel)

# ==== 🔎 Regex ====
def extract_site_code(text):
    match = re.search(r"[A-Z]{3,4}\d{3,4}", str(text))
    return match.group(0) if match else None

# ==== 📁 Build Lookup Dicts ====
excel_files = list(excel_dir.glob("*.xls*"))
ppt_files = list(excel_dir.glob("*.pptx"))

excel_lookup = {extract_site_code(f.stem): f for f in excel_files if extract_site_code(f.stem)}
ppt_lookup = {extract_site_code(f.stem): f for f in ppt_files if extract_site_code(f.stem)}

total_excel_files = len(excel_lookup)
zipped_sites_count = 0
summary_records = []

# ==== 🔁 Process Mapping Rows ====
for _, row in df.iterrows():
    site_code = str(row["New Site Code"]).strip()
    cluster_name = str(row["RF Cluster Name"]).strip()
    zip_name = str(row["File Name"]).strip() + '.zip'
    
    excel_file = excel_lookup.get(site_code)
    ppt_file = ppt_lookup.get(site_code)
    used_fallback = False
    fallback_code = None

    # === Fallback via DU ID ===
    if not excel_file or not ppt_file:
        du_id = str(row["DU ID"])
        fallback_code = extract_site_code(du_id)
        if fallback_code:
            if not excel_file:
                excel_file = excel_lookup.get(fallback_code)
                if excel_file:
                    used_fallback = True
                    print(f'🔁 Fallback matched DU ID for Excel: {fallback_code} → used for {site_code}')
            if not ppt_file:
                ppt_file = ppt_lookup.get(fallback_code)
                if ppt_file:
                    used_fallback = True
                    print(f'🔁 Fallback matched DU ID for PPT: {fallback_code} → used for {site_code}')

    # === Get Word File ===
    cluster_folder = cluster_dir / cluster_name
    word_files = list(cluster_folder.glob('*.docx')) if cluster_folder.exists() else []
    word_file = word_files[0] if word_files else None

    # === Check Files ===
    if not excel_file or not word_file:
        summary_records.append({
            'Site Code': site_code,
            'Cluster Name': cluster_name,
            'ZIP Name': zip_name,
            'Excel Included': bool(excel_file),
            'Word Included': bool(word_file),
            'PPT Included': bool(ppt_file),
            'Used Fallback': used_fallback,
            'Status': '❌ Skipped',
        })
        print(f'❌ Missing file(s) for: {site_code} - Cluster: {cluster_name}')
        continue

    # === Create ZIP ===
    zip_path = output_dir / zip_name
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        zipf.write(excel_file, arcname=excel_file.name)
        zipf.write(word_file, arcname=word_file.name)
        if ppt_file:
            zipf.write(ppt_file, arcname=ppt_file.name)

    zipped_sites_count += 1
    print(f'✅ Created ZIP: {zip_path.name}')

    summary_records.append({
        'Site Code': site_code,
        'Cluster Name': cluster_name,
        'ZIP Name': zip_path.name,
        'Excel Included': True,
        'Word Included': True,
        'PPT Included': bool(ppt_file),
        'Used Fallback': used_fallback,
        'Status': '✅ Zipped',
    })

# ==== 📊 Summary ====
print('
📊 Summary:')
print(f'📁 Total Excel files found: {total_excel_files}')
print(f'📦 Total sites zipped: {zipped_sites_count}')
print(f'❓ Unmatched sites (not zipped): {len(df) - zipped_sites_count}')

summary_df = pd.DataFrame(summary_records)
summary_path = output_dir / f'zip_summary_{date_str}.xlsx'

# Show cluster names where Word file was missing
missing_clusters = summary_df.loc[~summary_df['Word Included'], 'Cluster Name'].unique().tolist()
print('Clusters missing Word files:', missing_clusters)

display(summary_df)
